# Netflix Prize - Models and Evaluation

Workflow for the process of MODEL building & its EVALUATION:
1. time based train/test split
2. training of 3 models: baseline, item-based KNN, SVD
3. evaluating RMSE + MAE
4. evaluating MAP@10 properly (full catalog ranking, with relevance rating of >= 3.5 as per PS)
5. generate TOP 10 recommendations for some example users
6. looking at success and failure cases

In [13]:
# Importing necessary libraries
import time
import numpy as np
import pandas as pd

from surprise import Dataset, Reader, SVD, KNNWithMeans, BaselineOnly, accuracy

# DATASET info
data_dir = "../data"
relevance_cutoff = 3.5
top_k = 10
test_frac = 0.2

# MAP over full catalog is slow on my RAM, so evaluating on a 1000 user sample
n_eval_users = 1000   
seed = 42
np.random.seed(seed)

In [14]:
# Loading the RATINGS
df = pd.read_parquet(f"{data_dir}/ratings_model.parquet")

# Info about movie titles
def get_movie_titles(path):
    rows = []
    with open(path,"r",encoding="latin1") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            movie_id, year, title = line.split(",",2)
            rows.append((int(movie_id),int(year) if year != "NULL" else np.nan,title))
    return pd.DataFrame(rows,columns=["movie_id","year","title"])

# Getting movie titles
movie_titles = get_movie_titles(f"{data_dir}/movie_titles.csv").set_index("movie_id")

print(len(df), "ratings,", df["user_id"].nunique(), "users,", df["movie_id"].nunique(), "movies")

3438319 ratings, 50000 users, 4499 movies


## 1. Train/test split:

I used a time based split per user for my MODEL: oldest 80% of each user's ratings goes to train, newest 20% to test

**Why not a random split:**
- the EDA that we did at the initial stage, showed that the rating behaviour changes over time (mean rating jumps around 2004)
- So, a random split would let the model train on a user's future ratings to predict their past, which is leakage
- a time split will be most suitable as it will predict the future from the past

**Why per user and not one global date:**
- with a global date, users who joined late would be completely unseen in training and the model could only predict the global average for them
- that would mix the cold-start problem into the metrics
- Per user split means that every test user has some training history

In [15]:
# Splitting the DATASET
def split_by_time(df,test_frac=0.2):
    
    # Oldest ratings per user -> train, Newest -> test
    df = df.sort_values(["user_id","date"]).reset_index(drop=True)
    rank = df.groupby("user_id").cumcount()
    n = df.groupby("user_id")["rating"].transform("size")
    test_mask = rank >= np.ceil(n*(1 - test_frac))
    return df[~test_mask].copy(), df[test_mask].copy()

train_data, test_data = split_by_time(df, test_frac)

# Checking for leakage
check = (train_data.groupby("user_id")["date"].max().to_frame("max_train")
         .join(test_data.groupby("user_id")["date"].min().to_frame("min_test"),how="inner"))
print("users with leakage:",(check["max_train"] > check["min_test"]).sum())
print("train:",len(train_data)," test:",len(test_data))

users with leakage: 0
train: 2770037  test: 668282


In [16]:
# Converting to surprise format
reader = Reader(rating_scale=(1, 5))
trainset = (Dataset.load_from_df(train_data[["user_id","movie_id","rating"]],reader)
            .build_full_trainset())
testset = list(test_data[["user_id","movie_id","rating"]].itertuples(index=False,name=None))

## 2. Models:

**1. BaselineOnly**: it predicts global mean + user bias + item bias (just using it to show a comparison)

**2. KNNWithMeans (Item based):** 
- classic collaborative filtering
- ITEM based is used instead of USER based because the (item x item) similarity matrix is much smaller
- WithMeans because the EDA we did show that users rate on different scales

**3. SVD:**
- matrix factorization, the approach that Netflix Prize made famous
- it learns latent vectors for users and movies + bias terms

*I tried a few settings for n_factors and reg_all manually and kept the best ones*

In [19]:
# Baseline model

# For time
t0 = time.time()
baseline_model = BaselineOnly(verbose=False)
baseline_model.fit(trainset)

# Checking fit time
print("fit time:",round(time.time() - t0,1),"s")

pred_baseline = baseline_model.test(testset)

# RMSE & MAE metrics
print("RMSE:",round(accuracy.rmse(pred_baseline,verbose=False),4))
print("MAE:",round(accuracy.mae(pred_baseline,verbose=False),4))

fit time: 3.8 s
RMSE: 0.925
MAE: 0.7256


In [21]:
# Item based (KNN)

# For time
t0 = time.time()
knn_model = KNNWithMeans(k=40,
                         sim_options={"name":"pearson_baseline","user_based": False},
                         verbose=False)
knn_model.fit(trainset)

# Checking fit time
print("fit time:",round(time.time() - t0,1),"s")

t0 = time.time()

# Checking pred. time
pred_knn = knn_model.test(testset)
print("pred time:",round(time.time() - t0,1),"s")

# RMSE & MAE metrics
print("RMSE:",round(accuracy.rmse(pred_knn,verbose=False),4))
print("MAE:",round(accuracy.mae(pred_knn,verbose=False),4))

fit time: 23.6 s
pred time: 34.5 s
RMSE: 0.8959
MAE: 0.6877


In [22]:
# SVD matrix factorization

# For time
t0 = time.time()

# Using epochs with learning rate 
svd_model = SVD(n_factors=100,n_epochs=25,lr_all=0.005,reg_all=0.05,random_state=seed)
svd_model.fit(trainset)

# Checking fit time
print("fit time:",round(time.time() - t0,1),"s")

t0 = time.time()

# Checking pred. time
pred_svd = svd_model.test(testset)
print("pred time:",round(time.time() - t0,1),"s")

# RMSE & MAE metrics
print("RMSE:",round(accuracy.rmse(pred_svd,verbose=False),4))
print("MAE:",round(accuracy.mae(pred_svd,verbose=False),4))

fit time: 15.8 s
pred time: 2.9 s
RMSE: 0.8885
MAE: 0.6923


**Observations:**

Details of the comparison (exact numbers in the report):
- both models beat the Baseline model on RMSE and MAE metrics & overall SVD matrix factorization is the best

**KEY INSIGHT:**
- ITEM based (KNN) has the better MAE but worse RMSE than SVD
1. KNN is slightly better on typical predictions but makes a few big mistakes (probably items with few common users who rate)
2. SVD's errors are more controlled because of regularization

**Efficiency:**
- SVD predicts way faster than KNN (less fit time)
- KNN also needs the (item x item) similarity matrix, SVD scales linearly with the number of ratings
  
So for a REAL system SVD is the **more practical choice**

## 3. MAP@10:

To evaluate the ranking quality of our recommendation system, I used Mean Average Precision at 10 (MAP@10)

The metric is computed using the following approach:

- A movie is considered relevant if its test rating is greater than or equal to 3.5
- For each user, all unseen movies are ranked according to the predicted scores generated by the model
- The TOP 10 recommended movies are selected for evaluation
- Average Precision (AP@10) is calculated by considering the precision at every position where a relevant movie appears in the recommendation list

AP@10 = sum of precision at the rank at each hit/min(10, number of relevant test movies)

- Users with no relevant movies in the test set are excluded from the calculation
- MAP@10 is obtained by averaging the AP@10 scores across all evaluated users
- To reduce the computational cost, I performed the evaluation on a random sample of users while keeping the same sample for all models to ensure a fair comparison

**A higher MAP@10 score indicates that relevant movies are ranked closer to the top of the recommendation list**

In [23]:
# Info about movies
all_movies = set(train_data["movie_id"].unique())
user_rated_movies = train_data.groupby("user_id")["movie_id"].apply(set).to_dict()
movie_popularity = train_data["movie_id"].value_counts()

# Relevant test movies per user
relevant_movies = (test_data[test_data["rating"] >= relevance_cutoff].groupby("user_id")["movie_id"].apply(set).to_dict())
relevant_movies = {u:m & all_movies for u,m in relevant_movies.items()}
relevant_movies = {u:m for u,m in relevant_movies.items() if len(m) > 0}

# For ranking evaluation
eval_users = np.random.choice(list(relevant_movies.keys()),size=min(n_eval_users,len(relevant_movies)),replace=False)
print(len(eval_users),"users for ranking evaluation")

1000 users for ranking evaluation


In [24]:
# For recommendations
def calc_ap(ranked,relevant,k=top_k):
    hits = 0
    score = 0.0
    for rank, movie in enumerate(ranked[:k],start=1):
        if movie in relevant:
            hits += 1
            score += hits/rank
    return score/min(k,len(relevant))

#  Getting TOP movies
def get_top_movies(model,user,k=top_k):
    candidates = list(all_movies - user_rated_movies.get(user,set()))
    scores = np.array([model.predict(user,m).est for m in candidates])
    return [candidates[i] for i in np.argsort(-scores)[:k]]

def calc_map(model,users,k=top_k):
    aps = []
    for u in users:
        if model == "popularity":
            candidates = all_movies - user_rated_movies.get(u,set())
            ranked = sorted(candidates,key=lambda m: -movie_popularity.get(m,0))[:k]
        else:
            ranked = get_top_movies(model,u,k)
        aps.append(calc_ap(ranked,relevant_movies[u], k))
    return float(np.mean(aps))

In [25]:
# MAP@10 evaluation for Baseline
print("MAP@10 popularity baseline:", round(calc_map("popularity",eval_users),4))

# For time
t0 = time.time()

# MAP@10 evaluation for ITEM based (KNN)
print("MAP@10 KNN:",round(calc_map(knn_model,eval_users),4),f"({time.time()-t0:.0f}s)")

t0 = time.time()

# MAP@10 evaluation for SVD
print("MAP@10 SVD:",round(calc_map(svd_model,eval_users),4),f"({time.time()-t0:.0f}s)")

MAP@10 popularity baseline: 0.0439
MAP@10 KNN: 0.0008 (270s)
MAP@10 SVD: 0.0094 (28s)


**Observations from MAP@10 Evaluation:**

- Both ITEM based (KNN) & SVD models lose badly to the popularity baseline, even though they won at RMSE, this shows that the ranking performance did not improve proportionally
- This highlights an important characteristic of recommendation systems: a model that predicts ratings accurately does not always produce the best recommendation rankings

One possible reason is that certain movies with relatively few ratings receive high predicted scores, causing them to appear frequently in the top recommendations

**It shows that, ranking-based metrics such as MAP@10 may not always align with rating-prediction metrics like RMSE**

It is also important to note that offline evaluation metrics have limitations:
- A recommended movie may be relevant to a user even if it does not appear in the test data

**Therefore, MAP@10 should be interpreted alongside other metrics and qualitative recommendation examples**

## 4. Fixing the ranking popularity filter:

- I did many experimentations to find out what is the problem with my model's ranking behaviour
- Finally decided to tune some parameters to reduce this MAP@10 evaluation uncertainity

Idea: 
- only considered movies with at least `min_pop` training ratings as candidates, then personalized it within that pool
- sweeping min_pop to see the effect
(also printing pool size - if the pool gets tiny we are basically back to the popularity list)

In [26]:
# Fixing the filter using 'min_pop'
def get_top_movies_v2(model,user,k=top_k,min_pop=300):
    candidates = [m for m in all_movies - user_rated_movies.get(user,set())
                  if movie_popularity.get(m,0) >= min_pop]
    scores = np.array([model.predict(user,m).est for m in candidates])
    return [candidates[i] for i in np.argsort(-scores)[:k]]

def calc_map_v2(model,users,k=top_k,min_pop=300):
    aps = []
    for u in users:
        ranked = get_top_movies_v2(model,u,k,min_pop)
        aps.append(calc_ap(ranked,relevant_movies[u],k))
    return float(np.mean(aps))

# Sweeping 'min_pop'
for min_pop in [100,300,1000,3000,5000,8000,12000]:
    pool = int((movie_popularity >= min_pop).sum())
    score = calc_map_v2(svd_model,eval_users,min_pop=min_pop)
    print(f"min_pop={min_pop:>6} | pool={pool:>5} movies | SVD MAP@10={score:.4f}")

min_pop=   100 | pool= 1734 movies | SVD MAP@10=0.0107
min_pop=   300 | pool= 1065 movies | SVD MAP@10=0.0152
min_pop=  1000 | pool=  527 movies | SVD MAP@10=0.0273
min_pop=  3000 | pool=  234 movies | SVD MAP@10=0.0494
min_pop=  5000 | pool=  140 movies | SVD MAP@10=0.0575
min_pop=  8000 | pool=   81 movies | SVD MAP@10=0.0669
min_pop= 12000 | pool=   38 movies | SVD MAP@10=0.0608


In [27]:
# Final comparison at the best min_pop value
best_min_pop = 8000
print("MAP@10 at min_pop =",best_min_pop)
print("popularity baseline:",round(calc_map("popularity",eval_users),4))
print("KNN:",round(calc_map_v2(knn_model,eval_users,min_pop=best_min_pop),4))
print("SVD:",round(calc_map_v2(svd_model,eval_users,min_pop=best_min_pop),4))

MAP@10 at min_pop = 8000
popularity baseline: 0.0439
KNN: 0.0643
SVD: 0.0669


### Effect of Popularity Threshold on MAP@10:

- The popularity threshold experiment shows that recommendation quality initially improves as very low interaction movies are removed from the candidate pool
- MAP@10 increased steadily and reached its highest value around a minimum popularity threshold of **8,000 ratings** with **SVD MAP@10=0.0669**

- When the threshold was increased further to **12,000 ratings**, performance decreased to **SVD MAP@10=0.0608**
- At this point, only a small subset of highly popular movies (pool = 38 movies) available for recommendation, reducing the model's ability to provide personalized suggestions

- This observation highlights a trade-off between recommendation accuracy and dataset coverage
- Moderate filtering helps remove noisy and rarely rated movies, while excessive filtering limits the diversity of available recommendations

- The best performance was achieved at a popularity threshold of approximately **8,000 ratings**, where the balance between recommendation quality and movie coverage was most favorable

**Note:** The popularity threshold was selected using the same evaluation data used for performance measurement. A more rigorous approach would involve selecting this parameter on a separate validation set before final testing

## 5. Precision, Recall & Coverage:

- Additional metrics to test our model

In [28]:
def calc_precision_recall(model,users,k=top_k,min_pop=None):
    precisions, recalls, recommended = [], [], set()
    for u in users:
        if model == "popularity":
            candidates = all_movies - user_rated_movies.get(u,set())
            ranked = sorted(candidates,key=lambda m: -movie_popularity.get(m,0))[:k]
        elif min_pop is not None:
            ranked = get_top_movies_v2(model,u,k,min_pop)
        else:
            ranked = get_top_movies(model,u,k)
        hits = len(set(ranked) & relevant_movies[u])
        
        # For precision
        precisions.append(hits/k)
        
        # For recall
        recalls.append(hits/len(relevant_movies[u]))
        recommended.update(ranked)
    
    # For coverage    
    coverage = len(recommended)/len(all_movies)
    return np.mean(precisions), np.mean(recalls), coverage

for label, model, mp in [("popularity","popularity",None),
                         ("SVD raw",svd_model,None),
                         ("SVD min_pop=8000",svd_model,8000)]:
    p, r, c = calc_precision_recall(model,eval_users,min_pop=mp)
    print(f"{label:<18} precision@10={p:.4f} recall@10={r:.4f} coverage={c:.3f}")

popularity         precision@10=0.0734 recall@10=0.1128 coverage=0.017
SVD raw            precision@10=0.0192 recall@10=0.0177 coverage=0.062
SVD min_pop=8000   precision@10=0.0936 recall@10=0.1261 coverage=0.017


### Coverage Analysis:

The results highlight the trade-off between recommendation accuracy and catalog coverage

| Model | Precision@10 | Recall@10 | Coverage |
|--------|--------|--------|--------|
| Popularity Baseline | 0.0734 | 0.1128 | 0.017 |
| SVD (Raw) | 0.0192 | 0.0177 | 0.062 |
| SVD (min_pop = 8000) | 0.0936 | 0.1261 | 0.017 |

- The popularity baseline produced competitive results, achieving a Precision@10 of (**0.0734**) and Recall@10 of (**0.1128**). However, its coverage remained low (**0.017**), indicating that recommendations were concentrated among a small set of highly popular movies

- The raw SVD model achieved the highest catalog coverage (**0.062**), meaning recommendations were drawn from a much larger portion of the movie catalog. However, this came at the cost of lower recommendation quality, with a Precision@10 of only (**0.0192**) and Recall@10 of (**0.0177**)

- Applying a popularity threshold of (**8,000 ratings**) significantly improved ranking performance. The filtered SVD model achieved the highest Precision@10 (**0.0936**) and Recall@10 (**0.1261**), which is better than both the raw SVD model and the popularity baseline

**These results suggest that filtering out very low popularity movies helps by improving recommendations, although it reduces the diversity of movies that can be recommended**

## 6. Example Recommendations:

To understand the behavior of the recommendation system, recommendation lists are shown for three representative users with different rating patterns:

- A highly active user with a large rating history (active user)
- A moderately active user (moderate user)
- A user who generally assigns lower ratings (aggressive rater)

Along with the predicted ratings, two additional attributes are included:

- **Training Popularity (train_pop):** Total number of ratings received by the movie in the training data
- **Average Training Rating (train_mean):** Mean rating of the movie in the training data

These attributes help explain whether the model tends to recommend highly popular movies or less frequently rated ones. A recommendation is considered a **hit** if the user assigned a rating of 4 or higher to that movie in the test set.

In [30]:
# INFO of user
user_stats = train_data.groupby("user_id").agg(n_ratings=("rating","size"),mean_rating=("rating","mean"))
eligible = user_stats[user_stats.index.isin(relevant_movies)]

# INFO of movies
movie_stats = train_data.groupby("movie_id").agg(train_pop=("rating","size"),train_mean=("rating","mean"))

# Representative users
example_users = {
    "active user": eligible["n_ratings"].idxmax(),
    "moderate user": eligible[eligible["n_ratings"] <= eligible["n_ratings"].quantile(0.1)].index[0],
    "aggressive rater": eligible["mean_rating"].idxmin(),
}

# Recommendations
def show_recommendations(uid,label,model):
    print("=" * 80)
    print(label.upper(), f"(user {uid}: {user_stats.loc[uid, 'n_ratings']} train ratings,",
          f"mean given = {user_stats.loc[uid,'mean_rating']:.2f})")

    hist = (train_data[train_data["user_id"] == uid].nlargest(5,"rating")
            .join(movie_titles,on="movie_id"))
    print("\ntheir top rated movies:")
    for _, r in hist.iterrows():
        print(f" {r['rating']}* {r['title']}")

    recs = get_top_movies(model,uid)
    out = (pd.DataFrame({"movie_id": recs,
                         "pred": [model.predict(uid,m).est for m in recs]})
           .join(movie_titles,on="movie_id")
           .join(movie_stats,on="movie_id"))
    out["hit"] = out["movie_id"].isin(relevant_movies.get(uid,set()))
    print("\nTOP 10 recommendations (SVD,no popularity filter):")
    display(out[["title","pred","train_pop","train_mean","hit"]].round(2))

for label, uid in example_users.items():
    show_recommendations(uid,label,svd_model)

ACTIVE USER (user 2118461: 3016 train ratings, mean given = 4.18)

their top rated movies:
 5* Pay It Forward
 5* The Wedding Planner
 5* Beverly Hills Cop
 5* 50 First Dates
 5* Something's Gotta Give

TOP 10 recommendations (SVD,no popularity filter):


,title,pred,train_pop,train_mean,hit
0,Inu-Yasha,5.00,181,4.66,False
1,Lost: Season 1,5.00,177,4.73,False
2,Inu-Yasha: The Movie 3: Swords of an Honorable...,5.00,24,4.54,True
3,Mobile Suit Gundam SEED,4.99,56,4.59,False
4,Nip/Tuck: Season 2,4.95,251,4.40,False
5,Gentlemen of Fortune,4.94,17,4.41,False
6,Carlos Mencia: Not for the Easily Offended: Li...,4.93,22,4.32,False
7,"Empires: The Medici, Godfathers of the Renaiss...",4.91,61,4.23,False
8,GoodFellas: Special Edition: Bonus Material,4.90,52,4.27,False
9,My Fair Lady: Special Edition: Bonus Material,4.89,15,4.40,False


MODERATE USER (user 939: 8 train ratings, mean given = 4.12)

their top rated movies:
 5* Napoleon Dynamite
 5* Love Actually
 5* Buffalo '66
 4* PCU
 4* Army of Darkness

TOP 10 recommendations (SVD,no popularity filter):


,title,pred,train_pop,train_mean,hit
0,Six Feet Under: Season 4,4.89,462,4.42,False
1,Lost: Season 1,4.85,177,4.73,False
2,The West Wing: Season 3,4.85,467,4.48,False
3,The Simpsons: Season 6,4.82,482,4.50,False
4,Combat! Season 2: Mission 1,4.79,10,4.70,False
5,"The Godfather, Part II",4.79,8337,4.42,False
6,Lord of the Rings: The Fellowship of the Ring,4.78,20130,4.44,False
7,Monarch of the Glen: Series 2,4.76,162,4.26,False
8,Absolutely Fabulous: Series 5,4.76,476,4.22,False
9,Inspector Morse 15: Masonic Mysteries,4.73,103,4.20,False


AGGRESSIVE RATER (user 792248: 20 train ratings, mean given = 1.30)

their top rated movies:
 5* Alien: Collector's Edition
 3* The Legend 2
 1* Rush Hour 2
 1* The Mummy
 1* Dr. Dolittle 2

TOP 10 recommendations (SVD,no popularity filter):


,title,pred,train_pop,train_mean,hit
0,The West Wing: Season 3,3.09,467,4.48,False
1,The Simpsons: Season 3,3.07,2621,4.44,False
2,"The Godfather, Part II",3.01,8337,4.42,False
3,The Simpsons: Season 6,2.99,482,4.50,False
4,Curb Your Enthusiasm: Season 3,2.99,883,4.35,False
5,The Simpsons: Treehouse of Horror,2.97,1697,4.40,False
6,House of Cards Trilogy I: House of Cards,2.96,223,4.17,False
7,"Upstairs, Downstairs: Season 3",2.95,197,4.03,False
8,House of Cards Trilogy II: To Play the King,2.94,190,4.24,False
9,Star Trek: The Next Generation: Season 6,2.94,920,4.28,False


### Observations from Recommendation Examples:

The recommendation examples provide useful insights into both the success and failures of the SVD model

#### Successful Cases:

**1. Personalized Recommendations:**

- For the highly active user (3016 training ratings), the model successfully recommended *Inu-Yasha: The Movie 3*, which had only **24 ratings** in the training data. The user later rated this title positively in the test set (Hit = True).
- This demonstrates the model's ability to identify user specific interests and recommend less popular content that may not appear in a simple popularity based recommendation list.

**2. User Bias Modeling:**

- For the aggressive rater, whose average rating is only **1.30**, the model assigned recommendation scores between **2.9 and 3.1** instead of predicting very high ratings. This indicates that the SVD model successfully learns individual rating behavior and adjusts predictions according to each user's rating habits.


#### Limitations and Failure Cases:

**1. Sequential Content Problem:**

- For the moderate user, the model recommended titles such as *Six Feet Under: Season 4* and *Absolutely Fabulous: Series 5*.
- Since collaborative filtering relies only on rating patterns, it does not understand that content is sequential. As a result, later seasons may be recommended even when the user has not watched earlier seasons.

**2. Special Edition Recommendations:**

- Several recommendations correspond to limited editions, such as *GoodFellas: Special Edition: Bonus Material* and *My Fair Lady: Special Edition: Bonus Material*.
- These items often receive high ratings from a small but dedicated audience, causing them to receive unusually high predicted scores.

**3. Long-Tail Effects for Highly Active Users:**

- For users with extensive rating histories, many popular movies have already been consumed. Which results in, increased shift of recommendations toward less popular movies, which may reduce recommendation quality and increase the appearance of niche content.

**4. Limited Diversity:**

- The recommendation list for the aggressive rater contains multiple entries from the same series, including several *Simpsons* titles.
- This occurs because recommendations are generated independently for each item, without considering diversity across the final recommendation list.


#### Key Observation:

- An interesting trade-off was observed during popularity filtering as the popularity threshold that improved ranking metrics (min_pop = 8000) would also remove niche recommendations such as *Inu-Yasha: The Movie 3*.
- This highlights an important challenge in recommender systems: filtering low-popularity items can improve overall ranking performance but may also reduce the model's ability to discover personalized niche content.

## 7. Checking the niche-movie problem in numbers:

Checking how often the unfiltered SVD TOP-10s contain "niche" movies (fewer than 100 ratings but average above 4.3).

In [31]:
# Niche movies
niche_recs = []
for u in eval_users[:200]:
    for m in get_top_movies(svd_model,u):
        st = movie_stats.loc[m]
        if st["train_pop"] < 100 and st["train_mean"] > 4.3:
            niche_recs.append({"user_id":u,"movie_id":m,
                               "train_pop":int(st["train_pop"]),
                               "train_mean":round(float(st["train_mean"]), 2),
                               "title":movie_titles.loc[m, "title"]})

niche_df = pd.DataFrame(niche_recs)
print(f"niche movies in top-10s: {len(niche_df)} of {200 * top_k} slots ({len(niche_df)/(200*top_k):.1%})")
niche_df.head(15)

niche movies in top-10s: 201 of 2000 slots (10.1%)


,user_id,movie_id,train_pop,train_mean,title
0,461949,4041,56,4.59,Mobile Suit Gundam SEED
1,461949,3498,10,4.70,Combat! Season 2: Mission 1
2,461949,1230,22,4.32,Carlos Mencia: Not for the Easily Offended: Li...
3,563864,1109,15,4.40,My Fair Lady: Special Edition: Bonus Material
4,363252,4041,56,4.59,Mobile Suit Gundam SEED
5,363252,159,17,4.41,Gentlemen of Fortune
6,363252,1418,24,4.54,Inu-Yasha: The Movie 3: Swords of an Honorable...
7,109987,2568,72,4.44,Stargate SG-1: Season 8
8,56482,1418,24,4.54,Inu-Yasha: The Movie 3: Swords of an Honorable...
9,56482,4041,56,4.59,Mobile Suit Gundam SEED


### Impact of Niche Movies on Recommendations:

- Analysis of the recommendation lists revealed that **201 out of 2000 recommendation slots (10.1%)** were occupied by movies with very low training popularity.
- Several of these movies appeared repeatedly across recommendations for different users

Examples include:
- *Inu-Yasha: The Movie 3* (**24 training ratings**)
- *Combat! Season 2: Mission 1* (**10 training ratings**)
- *Gentlemen of Fortune* (**17 training ratings**) 
- *My Fair Lady: Special Edition: Bonus Material* (**15 training ratings**)

- The repeated appearance of the same low-popularity movies across multiple users suggests that the model may overestimate the relevance of certain niche items due to limited training data.
- As a result, these movies can occupy recommendation slots that might otherwise be used for wide relevant content

**This observation helped me to use the popularity filtering (`min_pop`), which reduced the influence of niche movies and improved ranking performance in later experiments**

## 8. Conclusions:

1. SVD gives the best RMSE, both models beat the bias-only baseline
2. Good RMSE does not mean good ranking - at full dataset of TOP-10,  both models lost to a simple popularity ranking because of the niche-movies
trap
3. After filtering candidates by popularity, SVD beats the popularity baseline approximately by 50% MAP@10 (personalization does add value but within the popular head)
4. Main limitations: 
- only file 1 of the dataset (~25% of catalog)
- users filtered to >= 10 ratings
- min_pop tuned on the test users
- no movie metadata used

5. Future work: 
- making a HYBRID model with metadata (fixes the season problem)
- keeping diversity in mind while working on such datasets